# Figure 04 -- wall-clock vs N

Loads `results/scaling/wallclock_vs_n.json`, produced by `bench/scaling/wallclock.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.repo_root() / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("scaling/wallclock_vs_n.json")
cfg, recs, fits = art["config"], art["data"]["records"], art["data"]["fits"]

SERIES = [
    ("jaccpot_potential", "jaccpot (potential)", "jaccpot"),
    ("jaxfmm_potential", "jaxFMM (potential)", "jaxfmm"),
    ("jaccpot_acceleration", "jaccpot (acceleration)", "jaccpot"),
    ("direct_acceleration", "direct sum $O(N^2)$", "direct"),
]
set_name = sorted({r["param_set"] for r in recs})[0]

fig, ax = style.figure(width=style.ONE_COL, height=2.9)
for i, (key, label, entity) in enumerate(SERIES):
    sel = sorted(
        (r for r in recs
         if r["param_set"] == set_name and r["timings"].get(key, {}).get("min_s")),
        key=lambda r: r["n"],
    )
    if not sel:
        continue
    fit = fits.get(f"{set_name}:{key}", {})
    alpha = fit.get("exponent")
    suffix = f"  ($\\alpha={alpha:.2f}$)" if alpha and np.isfinite(alpha) else ""
    ax.plot(
        [r["n"] for r in sel],
        [r["timings"][key]["min_s"] for r in sel],
        marker=style.MARKERS[i % len(style.MARKERS)],
        # Acceleration vs potential distinguished by dash, since two series share
        # the jaccpot colour by design (same code, different output).
        linestyle="--" if "acceleration" in key and entity == "jaccpot" else "-",
        color=style.entity_color(entity),
        label=label + suffix,
        markersize=3.4,
    )
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$N$")
ax.set_ylabel("wall-clock per evaluation [s]")
style.finish(ax, legend_kwargs={"loc": "upper left", "fontsize": 6.2})
style.footer(
    fig,
    jsonio.config_caption(cfg, ["order", "theta", "basis", "preset", "precision", "device", "seed"])
    + f"   exponent fitted for N >= {cfg.get('fit_min_n')}",
)
fig.tight_layout()
style.save(fig, FIG_DIR / "fig04_wallclock_vs_n.pdf")

for name, fit in sorted(fits.items()):
    if name.startswith(set_name):
        print(f"{name:<45s} alpha={fit['exponent']:.3f} R2={fit['r_squared']:.4f} "
              f"n={fit['n_points']} over N={fit.get('fit_min_n')}..{fit.get('fit_max_n')}")


## Caption

Wall-clock per force evaluation against $N$ on one A100, log-log, with fitted
power-law exponents $\alpha$ in the legend. The timed region is evaluation on an
already-built tree for every FMM series, which is what a simulation pays per step
when topology is reused. **jaccpot and jaxFMM are compared potential-to-potential**:
an acceleration costs strictly more than a potential, so the acceleration series
is shown separately (dashed) rather than timed against jaxFMM's output. The
$O(N^2)$ direct sum is cut off where it becomes unaffordable. Exponents are fitted
above the annotated $N$, below which a GPU is launch-overhead bound rather than
algorithm bound. Values from `results/scaling/wallclock_vs_n.json`.
